<a href="https://colab.research.google.com/github/karkhutmaria/KarkhutM_topical_compling_questions/blob/main/%D0%9A%D0%B0%D1%80%D1%85%D1%83%D1%82%D0%9C__04_homework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wikipedia sentence-transformers scikit-learn pandas numpy requests
import os
import time
import wikipedia
import numpy as np
import requests
import json
import re
from sentence_transformers import SentenceTransformer
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

  Preparing metadata (setup.py) ... done
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11756 sha256=20c80931df6e1a2be7ef8e933083cb35561db99cf01438f50ee395c6fd567f76
  Stored in directory: /root/.cache/pip/wheels/79/1d/c8/b64e19423cc5a2a339450ea5d145e7c8eb3d4aa2b150cde33b
Successfully built wikipedia


# Задание 1 (2 балла)


Выберите 5 языков в википедии (не тех, что использовались в семинаре). Скачайте по 10 случайных статей для каждого языка. Предобработайте тексты, удаляя лишние теги/отступы/разделители (если они есть). Разделите тексты на предложения и создайте датасет, в котором каждому предложению соответствует язык. Кластеризуйте тексты, используя эбмединг модель из прошлого семинара и любой алгоритм кластеризации. Проверьте качество кластеризации с помощь метрики ARI. Отдельно проанализируйте 3 ошибочно кластеризованных текста (если такие есть).

In [2]:
MIN_INTERVAL = 0.35
_last_call = [0.0]


def _wiki_request(params):
    params['format'] = 'json'
    params.setdefault('action', 'query')
    for attempt in range(5):
        wait = MIN_INTERVAL - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        _last_call[0] = time.time()
        r = requests.get(wikipedia.wikipedia.API_URL, params=params,
                         headers={'User-Agent': 'HSE-NLP-course-hw4/0.1 (karkhut.mari@gmail.com)'}, timeout=30)
        if r.status_code == 429:
            retry_after = int(r.headers.get('Retry-After', 30))
            print(f'429 Too Many Requests, ждем {retry_after} с')
            time.sleep(retry_after)
            continue
        r.raise_for_status()
        return r.json()
    r.raise_for_status()


wikipedia.wikipedia._wiki_request = _wiki_request

In [3]:
def load_with_disambigution(page):
    try:
        p = wikipedia.page(page)
    except wikipedia.DisambiguationError as e:
        random_option = np.random.choice(e.options)
        p = wikipedia.page(random_option)
    return p

In [4]:
def get_texts_for_lang(lang, n=100):
    wikipedia.set_lang(lang)
    wiki_content = []

    pages = wikipedia.random(n)

    for page_name in pages:
        try:
            page = load_with_disambigution(page_name)

        except Exception as e:
            print('Skipping page {}'.format(page_name), str(e).strip('\n'))
            continue

        wiki_content.append(f'{page.title}\n{page.content.replace("==", "")}')

    return wiki_content

In [5]:
code2lang = wikipedia.languages()
langs = ["da", "de", "nl", "no", "sv"]
[(lang, code2lang[lang]) for lang  in langs]

[('da', 'dansk'),
 ('de', 'Deutsch'),
 ('nl', 'Nederlands'),
 ('no', 'norsk'),
 ('sv', 'svenska')]

In [6]:
WIKI_CACHE = 'wiki_texts.json'

if os.path.exists(WIKI_CACHE):
    wiki_texts = json.load(open(WIKI_CACHE))
else:
    wiki_texts = {}

for lang in langs:
    if wiki_texts.get(lang):
        continue
    try:
        wiki_texts[lang] = get_texts_for_lang(lang, 10)
    except Exception as e:
        print('ERROR ON - ', lang, str(e).strip('\n'))
        continue

    print(lang, len(wiki_texts[lang]))
    time.sleep(1)

json.dump(wiki_texts, open(WIKI_CACHE, 'w'), ensure_ascii=False)

da 10


/usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.13/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


Skipping page Dzitnup "Dzitnup" may refer to: 
Dzitnup (Campeche)
Dzitnup (Yucatán)
Skipping page Sant’Angela Merici "Sant’Angela Merici" may refer to: 
Angela Merici
Sant’Angela Merici (Rom)
Sant’Angela Merici (Brescia)
Sant’Angela Merici (Mailand)
de 8
nl 10
Skipping page Elektrostatisk induksjon "Induksjon" may refer to: 
Induksjon i matematikken
Induksjon i filosofien og logikken
Elektromagnetisk induksjon
Elektrostatisk induksjon
Deduksjon
Induktans
no 9
sv 10


In [7]:
wiki_texts["da"][0]

'Clara Rao\nClara Rao (født 7. april 1984) er en dansk lokalpolitiker for Venstre, der har været borgmester i Allerød siden 1. januar 2026.\nEfter kommunalvalget 2025 blev Rao borgmester i Allerød efter en indgået konstitueringsaftale mellem Venstre, SF, Socialdemokratiet, Liberal Alliance og Radikale Venstre.\n\n\n Referencer '

In [8]:
def clean_wiki(text):
  cleaned = re.sub(r"\d+", " ", text)
  cleaned = re.sub(r"\s+", " ", cleaned)
  return cleaned

all_dictionaries = []
for key, value in wiki_texts.items():
  for val in value:
    clean = {key: clean_wiki(val)}
    for k, v in clean.items():
      split = re.split(r"(?<=[.!?…])\s+(?=[A-ZÆØÅÄÖÜẞÉ\"'»«„\u2018\u201c0-9])", v)
      for s in split:
        s = s.strip()
        if s:
          all_dictionaries.append({"language": key, "sentence": s})

print(all_dictionaries)

[{'language': 'da', 'sentence': 'Clara Rao Clara Rao (født . april ) er en dansk lokalpolitiker for Venstre, der har været borgmester i Allerød siden . januar .'}, {'language': 'da', 'sentence': 'Efter kommunalvalget blev Rao borgmester i Allerød efter en indgået konstitueringsaftale mellem Venstre, SF, Socialdemokratiet, Liberal Alliance og Radikale Venstre.'}, {'language': 'da', 'sentence': 'Referencer'}, {'language': 'da', 'sentence': 'Renato Perona Renato Perona (født . november i Terni, død . juli sammesteds) var en cykelrytter fra Italien, der primært kørte banecykling.'}, {'language': 'da', 'sentence': 'Perona indledte sin karriere som amatør på bane og landevej lige efter anden verdenskrig.'}, {'language': 'da', 'sentence': 'Perona blev udvalgt til at deltage i OL i London, hvor han kørte tandem sammen med Ferdinando Terruzzi.'}, {'language': 'da', 'sentence': 'De to italienere vandt først over et argentinsk par, hvorpå de i kvartfinalen besejrede et belgisk par.'}, {'language'

In [9]:
df = pd.DataFrame(all_dictionaries)
df.head()

,language,sentence
0,da,Clara Rao Clara Rao (født . april ) er en dans...
1,da,Efter kommunalvalget blev Rao borgmester i All...
2,da,Referencer
3,da,Renato Perona Renato Perona (født . november i...
4,da,Perona indledte sin karriere som amatør på ban...


In [10]:
def new_clean(text):
  cleaned = re.sub(r"\W+", " ", text)
  cleaned = re.sub(r"\s+", " ", cleaned)
  return cleaned

df["no_punct"] = df["sentence"].apply(new_clean)
df = df.drop("sentence", axis=1)
df.head()

,language,no_punct
0,da,Clara Rao Clara Rao født april er en dansk lok...
1,da,Efter kommunalvalget blev Rao borgmester i All...
2,da,Referencer
3,da,Renato Perona Renato Perona født november i Te...
4,da,Perona indledte sin karriere som amatør på ban...


In [11]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
embed = model.encode

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [12]:
X = model.encode(
    df["no_punct"].tolist(),
    normalize_embeddings=True,
    batch_size=64,
    show_progress_bar=True,
)

gold = df["language"].values
labels = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(X)

ari = adjusted_rand_score(gold, labels)
print(f"ARI: {ari:.3f}")

df["cluster"] = labels

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

ARI: 0.283


In [13]:
df.head(50)

,language,no_punct,cluster
0,da,Clara Rao Clara Rao født april er en dansk lok...,0
1,da,Efter kommunalvalget blev Rao borgmester i All...,0
2,da,Referencer,4
3,da,Renato Perona Renato Perona født november i Te...,0
4,da,Perona indledte sin karriere som amatør på ban...,0
5,da,Perona blev udvalgt til at deltage i OL i Lond...,0
6,da,De to italienere vandt først over et argentins...,0
7,da,Efter sejr over et schweizisk par i semifinale...,0
8,da,Finalen blev kørt i højst tre løb og briterne ...,0
9,da,På grund af forsinkelser i løbsprogrammet var ...,0


По метрике можно видеть, что модель довольно плохо справляется с кластеризацией родственных языков. У нее возникают большие трудности с тем, чтобы отличить датский язык от немецкого, хотя графически языки различаются довольно сильно. Также модель часто путает датский и норвежский, так как их письменная речь довольно схожа. Вероятно, кластеризация удалась бы лучше, если бы были собраны совершенно разыне языки.

## Задание 2 (2 балла)
Возьмите какой-нибудь из файлов из Syntagrus или Taiga -
https://github.com/UniversalDependencies/UD_Russian-SynTagRus/tree/master   
https://tatianashavrina.github.io/taiga_site/downloads  

Используя Dependency Matcher напишите 2 разных query к этому корпусу, результаты которых выдают что-то осмысленное (показывают какую-то закономерность; подвсечивают примеры какого-то феномена; выдают то, что можно использовать в другой задаче).
Добавьте дополнительные правила фильтрации, если получается слишком зашумленный или наоборот пустой результат (см. семинар и примеры для SVO)
Используйте именно синтактическую структуру, а не просто слова и POS тэги.
Обязательно покажите примеры, которые находятся по вашим запросам.


Ответ должен быть структурирован как-то так:
```
## query
Краткое описание того, что вы пытаетесь найти

### код
код с запросом (или несколько ячеек где вы последовательно улучшаете запрос)

### примеры
несколько самых показательных примеров или даже какая-то агрегация вроде топ-N

... (и так два раза)
```



In [18]:
!pip install conllu
import conllu
import spacy
from spacy.tokens import Doc
from spacy.matcher import DependencyMatcher
from conllu import parse
from io import open
from conllu import parse_incr
from collections import Counter

In [19]:
nlp_blank = spacy.blank('ru')

def conllu_to_doc(sentence):
    """CoNLL-U -> spaCy Doc (код из семинара)."""
    words = [t for t in sentence if isinstance(t['id'], int)]
    if not words:
        return None
    position = {t['id']: k for k, t in enumerate(words)}
    return Doc(
        nlp_blank.vocab,
        words=[t['form'] for t in words],
        spaces=[(t['misc'] or {}).get('SpaceAfter') != 'No' for t in words],
        heads=[position.get(t['head'], k) for k, t in enumerate(words)],
        deps=['ROOT' if t['head'] == 0 else t['deprel'] for t in words],
        pos=[t['upos'] for t in words],
        lemmas=[t['lemma'] for t in words],
        morphs=['|'.join(f'{k}={v}' for k, v in (t['feats'] or {}).items())
                for t in words],
    )

with open('ru_syntagrus-ud-train-a.conllu', encoding='utf-8') as f:
    sentences = conllu.parse(f.read())

docs = [d for d in map(conllu_to_doc, sentences) if d is not None]
print(f'Предложений: {len(docs)}')

Предложений: 24516


In [31]:
# query 1: безличные предложения с безличным глаголом

imp_matcher = DependencyMatcher(nlp_blank.vocab)
imp_matcher.add("FINITE_VERB", [[
    {"RIGHT_ID": "verb",
     "RIGHT_ATTRS": {
         "POS": "VERB",
         "MORPH": {"IS_SUPERSET": ["VerbForm=Fin"]},
     }},
]])


def is_clean_impersonal(doc, tok):
    if tok.dep_ != 'ROOT':
        return False

    for c in tok.children:
        if c.dep_ in ('nsubj', 'nsubj:pass', 'csubj', 'csubj:pass'):
            return False

    for c in tok.children:
        if c.dep_ in ('iobj', 'obl', 'expl') and 'Dat' in c.morph.get('Case'):
            return False

    m = tok.morph
    p, n = m.get('Person'), m.get('Number')
    g, t = m.get('Gender'), m.get('Tense')
    if not (('3' in p and 'Sing' in n) or ('Neut' in g and 'Past' in t)):
        return False

    for x in doc:
        if x.pos_ in ('NOUN', 'PROPN', 'PRON', 'ADJ', 'DET', 'NUM') \
           and 'Nom' in x.morph.get('Case'):
            return False
    return True


impersonal_hits = []
for doc in docs:
    for _, (vid,) in imp_matcher(doc):
        tok = doc[vid]
        if is_clean_impersonal(doc, tok):
            impersonal_hits.append((tok.lemma_, doc.text))

import random
random.seed(1)
seen, shown = set(), []
for l, s in impersonal_hits:
    if l in seen:
        continue
    seen.add(l)
    shown.append((l, s))
random.shuffle(shown)
for l, s in shown[:5]:
    print(f'Глагол:{l}\nКонтекст:{s}')

Глагол:хватать
Контекст:На что хватает энергии? 
Глагол:мочь
Контекст:Без воды не может? 
Глагол:сохраниться
Контекст:Во сне не сохранилось. 
Глагол:надиктовать
Контекст:Надиктует их, сидя в одиночестве или вместе с сыном Сергеем Никитичем перед магнитофоном. 
Глагол:пойти
Контекст:- Пошло! 


In [32]:
# query 2: расщепленный субъект

split_matcher = DependencyMatcher(nlp_blank.vocab)

split_matcher.add("DAT_WITH_SUBJ", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">",
     "RIGHT_ID": "dat",
     "RIGHT_ATTRS": {
         "DEP": {"IN": ["iobj", "obl", "expl"]},
         "MORPH": {"IS_SUPERSET": ["Case=Dat"]},
     }},
    {"LEFT_ID": "verb", "REL_OP": ">",
     "RIGHT_ID": "subj",
     "RIGHT_ATTRS": {"DEP": {"IN": ["nsubj", "nsubj:pass"]}}},
]])

split_matcher.add("DAT_NO_SUBJ", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">",
     "RIGHT_ID": "dat",
     "RIGHT_ATTRS": {
         "DEP": {"IN": ["iobj", "obl", "expl"]},
         "MORPH": {"IS_SUPERSET": ["Case=Dat"]},
     }},
]])

split_matcher.add("U_POSSESSIVE", [[
    {"RIGHT_ID": "verb", "RIGHT_ATTRS": {"POS": "VERB"}},
    {"LEFT_ID": "verb", "REL_OP": ">",
     "RIGHT_ID": "obl",
     "RIGHT_ATTRS": {
         "DEP": {"IN": ["obl", "iobj"]},
         "MORPH": {"IS_SUPERSET": ["Case=Gen"]},
     }},
    {"LEFT_ID": "obl", "REL_OP": ">",
     "RIGHT_ID": "u",
     "RIGHT_ATTRS": {"LEMMA": "у", "DEP": "case"}},
    {"LEFT_ID": "verb", "REL_OP": ">",
     "RIGHT_ID": "subj",
     "RIGHT_ATTRS": {"DEP": {"IN": ["nsubj", "nsubj:pass"]}}},
]])

def has_nominative(doc):
    for x in doc:
        if x.pos_ in ('NOUN', 'PROPN', 'PRON', 'ADJ', 'DET', 'NUM') \
           and 'Nom' in x.morph.get('Case'):
            return True
    return False

split_hits = []
for doc in docs:
    for mid, ids in split_matcher(doc):
        label = nlp_blank.vocab.strings[mid]

        if label == "DAT_WITH_SUBJ":
            vid, did, sid = ids
            verb, dat, subj = doc[vid], doc[did], doc[sid]

            if verb.lemma_ in {'дать', 'сказать', 'подарить', 'показать',
                               'сообщить', 'объяснить', 'принести', 'купить'}:
                continue
            split_hits.append({
                'type': 'DAT+SUBJ', 'verb': verb.lemma_,
                'dat': dat.text, 'subj': subj.text, 'sent': doc.text,
            })

        elif label == "DAT_NO_SUBJ":
            vid, did = ids
            verb, dat = doc[vid], doc[did]

            if has_nominative(doc):
                continue
            split_hits.append({
                'type': 'DAT (без Subj)', 'verb': verb.lemma_,
                'dat': dat.text, 'subj': '—', 'sent': doc.text,
            })

        elif label == "U_POSSESSIVE":
            vid, oid, uid, sid = ids
            verb, obl, u, subj = doc[vid], doc[oid], doc[uid], doc[sid]

            if verb.lemma_ not in {'быть', 'иметься', 'иметь'}:
                continue
            split_hits.append({
                'type': 'у+GEN', 'verb': verb.lemma_,
                'dat': obl.text, 'subj': subj.text, 'sent': doc.text,
            })

print(f'Всего вхождений: {len(split_hits)}')
for t in ('DAT+SUBJ', 'DAT (без Subj)', 'у+GEN'):
    print(f'  {t}: {sum(1 for h in split_hits if h["type"] == t)}')


print('\nДательный и именительный падежи')
shown = 0
for h in split_hits:
    if h['type'] != 'DAT+SUBJ':
        continue
    print(f'[{h["verb"]}: {h["dat"]} + {h["subj"]}]  {h["sent"]}')
    shown += 1
    if shown >= 5:
        break

print('\nДательный падеж без именительного')
shown = 0
for h in split_hits:
    if h['type'] != 'DAT (без Subj)':
        continue
    print(f'[{h["verb"]}: {h["dat"]}]  {h["sent"]}')
    shown += 1
    if shown >= 5:
        break

print('\nПредлог "у" + родительный падеж"')
shown = 0
for h in split_hits:
    if h['type'] != 'у+GEN':
        continue
    print(f'[{h["verb"]}: {h["dat"]} + {h["subj"]}]  {h["sent"]}')
    shown += 1
    if shown >= 5:
        break

Всего вхождений: 2656
  DAT+SUBJ: 1991
  DAT (без Subj): 488
  у+GEN: 177

Дательный и именительный падежи
[быть: вопросу + Я]  - Я была у вас неделю тому назад по вопросу устройства на работу. 
[отнестись: вопросу + Савельев]  Но Савельев несерьезно отнесся к вопросу. 
[казаться: Савельеву + этого]  Савельеву этого не казалось. 
[прибыть: нему + Ефимова]  Конечно, Савельеву легко было рассуждать, поскольку Ефимова прибыла к нему с направлением главка. 
[подобрать: вам + я]  А я вам уже подобрал место на периферии. 

Дательный падеж без именительного
[казаться: Савельеву]  Савельеву этого не казалось. 
[заказать: ему]  - Надо бы ему моторную заказать. 
[делать: мне]  Но что мне делать, прямо не знаю. 
[пойти: машине]  Потом зябко поежился и пошел к машине. 
[говорить: абонентам]  Говорят легковерным абонентам, что на складе нет материалов. 

Предлог "у" + родительный падеж"
[быть: него + лицо]  У него было полное симпатичное лицо некурящего человека и белые, совершенно седые волосы. 
[

## Задание 3 (4 балла)

Эссе

Прочитайте статью *FineWeb2: One Pipeline to Scale Them All — Adapting Pre-Training Data Processing to Every Language*.
Ссылка - https://arxiv.org/pdf/2506.20920

Напишите эссе, в котором вы, используя эту статью как конкретный пример, рассуждаете о том, что значит создать «хороший» многоязычный датасет для предобучения?

Не пересказывайте статью! Выберите что-то, что вам показалось интересным (assumption, результат эксперимента, метрику и т.п.), и постройте вокруг этого аргумент. Вы можете соглашаться с авторами, спорить с ними или добавлять что-то от себя.

Можете опираться на следующие вопросы, но можете и придумать что-то свое:

- Действительно ли возможен "one pipeline to scale them all", или некоторые аспекты отбора и очистки данных по своей природе должны быть специфичны для конкретного языка?
- Что именно означает "качество данных" в FineWeb2? Согласны ли в с этим?
- Если бы вам нужно было построить корпус для предобучения на языке, который не исследовался в статье, насколько вы бы доверяли предложенному пайплайну и где ожидали бы проблемы?
- Какое решение в дизайне FineWeb2 кажется вам наиболее убедительным — или, наоборот, наименее убедительным — и почему?
- FineWeb2 охватывает более тысячи языков, однако для многих низкоресурсных языков доступно очень мало данных, причём эти данные могут быть весьма однородными. Честно ли в этом случае заявлять об их покрытии? Как провести границу?
- В статье сначала удаляются дубликаты, а затем часть повторяющихся данных выборочно возвращается с помощью *rehydration*. То есть дубли это не всегда "плохие данные". Как тогда понять, что дедублицировать, а что нет?
- Авторы тщательно выбирают *early-signal* задачи для оценки, чтобы сравнивать разные варианты датасета. Достаточно ли качества модели на downstream-задачах для оценки качества корпуса?

Вы можете предлагать гипотетические примеры, альтернативные решения или эксперименты, с помощью которых можно было бы проверить ваш аргумент.

Цель задания — не покритиковать статью. Цель — построить связный аргумент, который нельзя было бы написать, не прочитав статью внимательно и не обдумав её содержание.



Статья с громким названием и забавной отсылкой на «Властелина колец», "One pipeline to scale them all", представляет новый пайплайн, в ходе которого собирается совершенно новый мультиязычный датасет. Этот датасет обещает быть достаточно гибким, чтобы подстроиться под малоресурсные языки. Авторы статьи утверждают, что модели, натренированные на нем, в большинстве случаев показывают лучшие метрики, чем прочие мультиязычные модели.

 Конечно, мультиязычный датасет звучит как настоящая мечта. В чем его плюс? Нам не нужно собирать разные языковые датасеты и отдельно тренировать на них модели. Достаточно скормить модели один огромный датасет, а дальше она сама разберется с тем, какой перед ней язык и как с ним работать. Это очень удобно с точки зрения распределения ресурсов, в первую очередь времени, которого всем всегда не хватает. Мультиязычный датасет позволит быстрее обучать и дообучать модели. Данные высокоресурсных языков можно будет экстраполировать на родственные им низкоресурсные языки, о чем и пишут авторы статьи.

 Что мне нравится в пайплайне FineWeb2, так это то, насколько последовательно авторы статьи его выстраивают. Они разбивают пайплайн на маленькие этапы (определение языка, дедупликация, фильтрация и восполнение датасета) и не просто выбирают одну стратегию поведения на данном конркетном этапе, а проверяют разные подходы применительно к разным языкам. Для одних языков лучше работает определенный токенизатор, для других необходимо осторожно подобрать порог прохождения данных в датасет. Мне очень понравился такой серьезный и гибкий подход. Однако и он не без изъяна.

 В силу того, что для подбора наилучших параметров для каждого языка требуется очень много времени и огромные вычислительные ресурсы, исследователи жестко ограничили свою подборку языков. Они экспериментировали на арабском, китайском, французском, хинди, русском, суахили, телугу, тайском и турецком. Результаты этих многоресурсных языков планировалось перенести на низкоресурсные языки. И в этом я вижу некоторую проблему. Авторы говорят о том, что выбрали эти языки, чтобы посмотреть на конкретные языковые семьи, но уже сейчас можно заметить проблему. Пайплайн не учитывает многие языковые семьи: германскую, австронезийскую, уральскую, алтайскую и др. Именно поэтому при оценке пайплайна на ранее не встречаемых моделью языках результаты для немецкого и индонезийского сильно проседают, нежели результаты для тех языков, что делят языковые семьи с языками обучения.

 Еще одна довольно значимая проблема, которая касается уже в частности самого мультиязычного датасета, это низкоресурсные языки как таковые. Дело в том, что все достаточно качественные тексты на этих языках это либо переводы Библии, либо статьи из Википедии. Библия - очень полезный ресурс при изучении малоресурсного языка, но обучать на ней модель не очень весело. Модель получается однобокой. А вот Википедия на малоресурсных языках нередко является продуктом машинного перевода, что делает ее довольно "грязным" зашумленным ресурсом, который не позволит добиться адекватного обучения модели.

 Это лишь только некоторые из проблем создания мультиязычного датасета и модели, но мне они кажутся самыми интересными.

 А что касаемо специфичных для языка датасетов и моделей? Уступают ли они многоязычным? Судя по метрикам, представленным в статье, совсем не уступают. Напротив, они зачастую справляются лучше, так как могут учесть всю специфику того или иного языка. Вот только их проблема в том, что качественный языковой датасет может получиться только для многоресурчных языков, когда модели есть на чем обучаться, и есть на чем проверяться. Именно поэтому важно понимать, что как многоязычным, так и специфичным для конкретного языка моделям есть место быть.

 Я считаю, что очень здорово, что получилось создать такой пайплайн и датасет, который позволит нам работать как с многоресурными, так и с малоресурсными языками в достаточном качестве. Я искренне надеюсь, что дальнейшие исследования смогут его усовершенствовать и как-то обойти обозначенные мной выше проблемы.